# Transformer Multivariate

In this section we implement multivariate forecasting using the Transformer Model with the **TimeSeriesDatasetVectorizedExog** approach.

The Transformer Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate, ...).

With this approach the model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. The self-attention mechanism captures dependencies across the entire sequence simultaneously.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Input Projection (input_size → d_model)
    ↓
Positional Encoding
    ↓
Transformer Encoder Layers (2 layers)
  ├─ Multi-Head Self-Attention (4 heads)
  ├─ Add & Norm
  ├─ Feedforward (d_model → 256 → d_model)
  └─ Add & Norm
    ↓
Global Average Pooling
    ↓
Dropout
    ↓
Fully Connected (d_model → 1)
    ↓
Output (1 prediction)
```

## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding for Transformer model.
    Adds information about the position of tokens in the sequence.
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, d_model)
        return x + self.pe[:, :x.size(1), :]


class TransformerForecaster(nn.Module):
    """
    Transformer model for MULTIVARIATE time series forecasting.
    Architecture: 
        Input Projection -> Positional Encoding -> 
        Transformer Encoder -> Global Average Pooling -> 
        Dropout -> Fully Connected
    
    Uses self-attention mechanism to capture dependencies.
    Can process entire sequence in parallel (unlike RNN/LSTM).
    """
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=256, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            d_model: Dimension of the model (must be divisible by nhead)
            nhead: Number of attention heads
            num_layers: Number of transformer encoder layers
            dim_feedforward: Dimension of feedforward network
            dropout: Dropout rate
        """
        super(TransformerForecaster, self).__init__()
        
        self.input_size = input_size
        self.d_model = d_model
        
        # Input projection: map input_size to d_model
        self.input_projection = nn.Linear(input_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # Important: batch_first=True for (batch, seq, feature) format
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        self.fc = nn.Linear(d_model, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Project input to d_model dimensions
        x = self.input_projection(x)  # (batch_size, seq_length, d_model)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Pass through transformer encoder
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_length, d_model)
        
        # Global average pooling over sequence dimension
        # Alternative: use last token or first token (like BERT's [CLS])
        pooled = transformer_out.mean(dim=1)  # (batch_size, d_model)
        
        # Apply dropout
        out = self.dropout(pooled)
        
        # Fully connected layer
        out = self.fc(out)  # (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

## Model Results with Exogenous Features